In [3]:
import pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline

df = pd.read_csv('train.csv')
df = df[['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']]

X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns=['Survived']), df['Survived'], test_size=0.2, random_state=42
)

# columns: 0 Pclass, 1 Sex, 2 Age, 3 SibSp, 4 Parch, 5 Fare, 6 Embarked
trf1 = ColumnTransformer([
    ('impute_age', SimpleImputer(), [2]),
    ('impute_embarked', SimpleImputer(strategy='most_frequent'), [6])
], remainder='passthrough')

# trf1 ke baad order: Age, Embarked, Pclass, Sex, SibSp, Parch, Fare
trf2 = ColumnTransformer([
    ('ohe_sex_embarked', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), [1, 3])
], remainder='passthrough')

trf3 = ColumnTransformer([('scale', MinMaxScaler(), slice(0, 10))])
trf4 = SelectKBest(score_func=chi2, k=8)
trf5 = DecisionTreeClassifier()

pipe = Pipeline([
    ('trf1', trf1), ('trf2', trf2), ('trf3', trf3), ('trf4', trf4), ('trf5', trf5)
])
pipe.fit(X_train, y_train)

pickle.dump(pipe, open('pipe.pkl', 'wb'))

# sir wala test
pipe = pickle.load(open('pipe.pkl', 'rb'))
test_input2 = np.array([2, 'male', 31.0, 0, 0, 10.5, 'S'], dtype=object).reshape(1, 7)
pipe.predict(test_input2)

C:\Users\Admin\.conda\envs\notebook-env\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(
C:\Users\Admin\.conda\envs\notebook-env\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(


array([0])